# Emotional Sinhala Speech Dataset — Kaggle pilot

Pilot of `build_emotional_sinhala_dataset.py` on **one** SLBC radio-drama episode
(*Muwan Palassa*) from the Internet Archive.

**Manifest-first:** this notebook produces `manifest.csv` + `report.json`, **not**
redistributed audio. The source items carry no license field — treat them as copyrighted.

## Before you run — Settings panel (right)

| Setting | Value |
|---|---|
| **Accelerator** | GPU T4 ×2 |
| **Internet** | ON |
| **Persistence** | **Files only** ← keeps `/kaggle/working` between sessions |
| **Add-ons → Secrets** | add `HF_TOKEN` = your HuggingFace token, and **attach it** |

## And on huggingface.co — accept the gated terms on BOTH pages

Logged in with the **same account** as your token:

1. <https://huggingface.co/pyannote/speaker-diarization-3.1>
2. <https://huggingface.co/pyannote/segmentation-3.0>  ← the pipeline's dependency; easy to miss

Miss either one and diarization silently degrades to a single `SPEAKER_UNK`, meaning your
clips are **not speaker-pure** — which defeats the point for TTS. Cell 5 below checks this
explicitly.

> **Run cells one at a time, top to bottom.** Do not use *Run All* — you want to stop at the
> review checkpoint before spending GPU quota on the full run.

In [ ]:
# --- 1. environment: pin one GPU + load the HF token from the Kaggle secret ----
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle gives T4 x2; pin to one GPU

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("OK: HF token loaded from Kaggle secret.")
except Exception as e:
    print("!! No HF_TOKEN secret:", e)
    print("!! Diarization will degrade to a single SPEAKER_UNK (clips NOT speaker-pure).")
    print("!! Fix: Add-ons -> Secrets -> add HF_TOKEN, and make sure it is attached.")

In [ ]:
# --- 2. get / update the pipeline code ----------------------------------------
%env GIT_TERMINAL_PROMPT=0
# ^ Makes git FAIL FAST instead of hanging forever on a hidden 'Username for
#   https://github.com:' prompt. If you DO see that prompt, the repo is private:
#   either make it public, or use the token variant at the bottom of this cell.
import os
REPO = "https://github.com/DSEgrp18/Dataset-creation-withEmotion.git"
os.chdir("/kaggle/working")
if not os.path.isdir("Dataset-creation-withEmotion"):
    !git clone $REPO
os.chdir("/kaggle/working/Dataset-creation-withEmotion")
!git pull --ff-only          # pick up the latest fixes if already cloned
!git log --oneline -1

# --- private repo? use a PAT stored as the Kaggle secret GITHUB_TOKEN ---------
# from kaggle_secrets import UserSecretsClient
# tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
# !git clone https://{tok}@github.com/DSEgrp18/Dataset-creation-withEmotion.git
#
# --- no GitHub at all? upload the .py as a Kaggle Dataset and copy it ---------
# !cp /kaggle/input/<your-dataset-name>/build_emotional_sinhala_dataset.py .

In [ ]:
# --- 3. install deps (~2-3 min) -----------------------------------------------
# torch / torchaudio / numpy / scipy are PREINSTALLED on Kaggle with CUDA wheels.
# Do NOT reinstall them -- everything below is additive.
!pip install -q internetarchive librosa soundfile pyloudnorm faster-whisper \
    "pyannote.audio>=3.1" demucs transformers
print("deps installed")

In [ ]:
# --- 4. stage 1 only: download (no GPU needed, ~30 s) -------------------------
# Cheap sanity check that networking + archive.org access work.
# Expect: 25.3 min, lang=sin, license=ABSENT (-> manifest-first).
!python build_emotional_sinhala_dataset.py --stage download --smoke \
    --identifiers muwan-palassa-140113 --work-dir /kaggle/working/eesd

In [ ]:
# --- 5. smoke: all 9 stages on the first ~3 min (~2 min on T4) ----------------
# Purpose: prove nothing CRASHES. Uses a tiny ASR model, so the Sinhala text will
# be poor and the usable yield is often 0 -- do NOT judge quality from this run.
#
# --force is included because re-running after a code fix must not reuse stale
# per-stage state (e.g. a diarize step that previously fell back to SPEAKER_UNK).
# It re-downloads the 24 MB mp3, which takes ~30 s. Drop it on a first clean run.
!python build_emotional_sinhala_dataset.py --stage all --smoke --force \
    --identifiers muwan-palassa-140113 --work-dir /kaggle/working/eesd

In [ ]:
# --- 6. VERIFY diarization actually ran ---------------------------------------
# This failure is silent by design (the pipeline continues), so check it explicitly.
import json, glob
speakers, turns = set(), 0
for p in glob.glob("/kaggle/working/eesd/meta/*.json"):
    meta = json.load(open(p))
    for sf in meta.get("source_files", []):
        for t in sf.get("diarization", []):
            speakers.add(t["speaker"])
            turns += 1
print("speakers found:", sorted(speakers), "| turns:", turns)
print()
if not speakers or speakers == {"SPEAKER_UNK"}:
    print("!! DIARIZATION DID NOT RUN -- clips are NOT speaker-pure.")
    print("   Accept the gated terms with the SAME HF account on BOTH:")
    print("     https://huggingface.co/pyannote/speaker-diarization-3.1")
    print("     https://huggingface.co/pyannote/segmentation-3.0")
    print("   then re-run cell 5 (it already has --force).")
else:
    print("OK:", len(speakers), "distinct speakers across", turns, "turns.")

### Checkpoint

Only continue when **cell 5 reaches `--- stage: manifest ---` without a traceback** and
**cell 6 reports more than one speaker**. Fixing problems now costs seconds; finding them
after the full run costs GPU quota.

In [ ]:
# --- 7. the real pilot: full episode, whisper large-v3 (~20-40 min on T4) ------
# Drops --smoke: full 25.3 min, real ASR, real emotion labels.
# Add --force ONLY if you are re-running this after another code change.
!python build_emotional_sinhala_dataset.py --stage all \
    --identifiers muwan-palassa-140113 --work-dir /kaggle/working/eesd

In [ ]:
# --- 8. read the yield funnel + emotion distribution --------------------------
import json, os
import pandas as pd

rep = json.load(open("/kaggle/working/eesd/report.json"))
print("FUNNEL (minutes):", json.dumps(rep.get("funnel_minutes", {}), indent=2))
print("USABLE YIELD %:", rep.get("yield_percent"))
print("COUNTS:", rep.get("counts", {}))

ed = rep.get("emotion_distribution", {})
cc = ed.get("category_counts", {})
if cc:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].bar(list(cc.keys()), list(cc.values()))
    ax[0].set_title("Emotion distribution (usable clips)")
    ax[0].tick_params(axis="x", rotation=30)
    ah = ed.get("arousal_hist", {})
    if ah:
        ax[1].bar(list(ah.keys()), list(ah.values()))
        ax[1].set_title("Arousal histogram")
        ax[1].tick_params(axis="x", rotation=60)
    plt.tight_layout(); plt.show()
else:
    print("No usable-clip emotion labels (expected on a tiny --smoke sample).")

for name in ("manifest.csv", "needs_manual_transcription.csv"):
    p = f"/kaggle/working/eesd/{name}"
    if os.path.exists(p):
        df = pd.read_csv(p)
        print()
        print(f"=== {name}: {len(df)} rows ===")
        print(df.head(10).to_string())

In [ ]:
# --- 9. listen to a few clips before you trust anything -----------------------
# Spot-check that the audio is clean speech and the Sinhala text roughly matches.
import pandas as pd
from IPython.display import Audio, display

df = pd.read_csv("/kaggle/working/eesd/manifest.csv")
for _, r in df.head(5).iterrows():
    print(f"[{r.emotion_label}] a={r.arousal} v={r.valence} asr={r.asr_conf}  {r.text}")
    display(Audio(f"/kaggle/working/eesd/clips/{r.clip_id}.wav"))

In [ ]:
# --- 10. SAVE the dataset before the session dies -----------------------------
# /kaggle/working is wiped between sessions unless Persistence = 'Files only'.
# These three files ARE the dataset (manifest-first). Download them from the
# right-hand Data/Output panel, and commit them to the repo -- never the audio.
import shutil, os
for name in ("manifest.csv", "needs_manual_transcription.csv", "report.json",
             "download_manifest.csv"):
    src = f"/kaggle/working/eesd/{name}"
    if os.path.exists(src):
        shutil.copy(src, f"/kaggle/working/{name}")
        print("ready to download:", name, os.path.getsize(src), "bytes")

## Review, then scale — do NOT auto-download the whole archive

Read the funnel in cell 8 before adding any more episodes:

- **`yield_percent`** — expect **<10–20 %**. 5–15 % is normal and healthy. ~0 % means
  something is misconfigured, not that the data is bad.
- **`funnel_minutes`** — shows *where* audio died (separate → diarize → VAD → filter).
- **`emotion_distribution`** — if it is nearly all `neutral`/`calm_content`, the expressive
  clips are being lost, most likely to ASR failure.
- **`needs_manual_transcription.csv`** — the high-arousal clips that failed ASR. These are
  the ones you most want for expressive TTS; they are kept for a human pass, not discarded.

**If yield is too low**, loosen the filters rather than accepting the loss:

```bash
--min-asr-conf 0.4 --min-align-conf 0.4 --min-snr-db 5
```

**Then scale** (idempotent — processed episodes are skipped, so you can add more across
sessions):

```bash
!python build_emotional_sinhala_dataset.py --stage all \
    --identifiers muwan-palassa-210113,MuwanPalassa29816,muwanpalassa_27513 \
    --work-dir /kaggle/working/eesd
```

Or everything: `--search 'title:("Muwan Palassa")'`. Watch the 30 h GPU quota — a full
25-min episode costs roughly 20–40 min of it.